In [3]:
!ls ..

nr_dose  nr_dose_scratch


In [ ]:
from monai.transforms import Compose,LoadImaged, EnsureChannelFirstd, ScaleIntensityRanged, Spacingd, ResizeWithPadOrCropd
from scripts.data_loader import ExtractSlabsAroundZ, InjectGaussianBeamPriord
from Omegaconf import OmegaConf

In [26]:
dataPath = "/project/nr_dose_scratch/training"
sample = "1ABB011"

In [30]:
!ls $dataPath

1ABB006  1ABB070  1ABB143  1THB023  1THB076  1THB217
1ABB011  1ABB078  1ABB145  1THB027  1THB078  1THB218
1ABB020  1ABB083  1ABB147  1THB029  1THB095  1THB220
1ABB021  1ABB098  1ABB149  1THB031  1THB119  1THB221
1ABB030  1ABB102  1ABB155  1THB037  1THB120  1THB226
1ABB031  1ABB109  1ABB161  1THB043  1THB121  SynthRAD2025_image_parameters.xlsx
1ABB035  1ABB110  1ABB164  1THB045  1THB122  beam_parameters.json
1ABB036  1ABB115  1ABB169  1THB048  1THB143  train.json
1ABB039  1ABB118  1THB002  1THB052  1THB191  val.json
1ABB041  1ABB123  1THB008  1THB054  1THB195
1ABB042  1ABB124  1THB011  1THB058  1THB202
1ABB045  1ABB128  1THB016  1THB063  1THB205
1ABB061  1ABB135  1THB017  1THB067  1THB211
1ABB067  1ABB138  1THB021  1THB074  1THB214


In [28]:
sample = "1ABB011"
transforms = Compose([
    LoadImaged(keys=["ct"], image_only=False),
])
sample_dict = {"ct": f"{dataPath}/{sample}/image/ct.mha"}
data = transforms(sample_dict)

In [ ]:
cfg = OmegaConf.load("configs/default_config.yaml")
cfg = cfg['komondor']

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["ct"],image_only=False),
    EnsureChannelFirstd(keys=["ct"]),
    ScaleIntensityRanged(keys=['ct'],a_min=cfg['ct_min'], a_max=cfg['ct_max'], b_min=0.0, b_max=1.0, clip=True),
    ExtractSlabsAroundZ(keys=["ct", "gt_dose"], source_key="ray_source", slice_radius=15),
    Spacingd(keys=["ct"], pixdim=cfg['pixdim'], mode='trilinear'), # pixdim = [4.0, 4.0, 3.0]
    ResizeWithPadOrCropd(keys=["ct",], spatial_size=cfg['roi_size']), #roi_size = [128, 128, 32] 
    InjectGaussianBeamPriord(keys =['ct'],source_key="ray_source", target_key="ray_target", ref_key="ct", sigma=cfg['sigma'], flip_lps_to_ras = True),
    ])

In [ ]:
data = train_transforms(sample_dict)

{'ct': metatensor([[[-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          ...,
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.]],
 
         [[-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          ...,
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.]],
 
         [[-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -1024.,  ..., -1024., -1024., -1024.],
          [-1024., -1024., -10